# Food RAG — Step 2: Build Index (GPU)

**執行前先確認：Runtime → Change runtime type → T4 GPU**

流程：
1. 安裝套件
2. 上傳 `chunks.jsonl`、`violations.jsonl`、`failed_files.jsonl`
3. 用 BGE-M3 編碼（GPU 加速）
4. 建 FAISS index + SQLite
5. 下載三個輸出檔回本機

In [ ]:
# ① 確認 GPU
import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  沒有 GPU！請到 Runtime → Change runtime type → T4 GPU")

In [ ]:
# ② 安裝套件
!pip install -q faiss-cpu sentence-transformers tqdm numpy

In [ ]:
# ③ 上傳檔案
# 請選取本機 data/processed/ 裡的三個檔案
from google.colab import files

print("請上傳以下三個檔案（可一次多選）:")
print("  - chunks.jsonl")
print("  - violations.jsonl")
print("  - failed_files.jsonl")
uploaded = files.upload()
print(f"\n已上傳: {list(uploaded.keys())}")

In [ ]:
# ④ 主程式
import json
import sqlite3
from pathlib import Path

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# ── 設定 ──────────────────────────────────────────
EMBED_MODEL  = "BAAI/bge-m3"
EMBED_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EMBED_BATCH  = 64      # GPU 上可以開大
DB_PATH      = Path("chunks.db")
FAISS_CHUNKS = Path("faiss_chunks.index")
FAISS_CASES  = Path("faiss_cases.index")

print(f"使用裝置: {EMBED_DEVICE}")

# ── Schema ────────────────────────────────────────
SCHEMA_SQL = """
DROP TABLE IF EXISTS chunks;
DROP TABLE IF EXISTS chunk_laws;
DROP TABLE IF EXISTS violations;
DROP TABLE IF EXISTS failed_files;

CREATE TABLE chunks (
    id            INTEGER PRIMARY KEY AUTOINCREMENT,
    text          TEXT NOT NULL,
    category      TEXT,
    primary_law   TEXT,
    subtopic      TEXT,
    document      TEXT,
    kind          TEXT,
    is_ocr        INTEGER DEFAULT 0,
    has_table     INTEGER DEFAULT 0,
    char_len      INTEGER,
    chunk_strategy TEXT,
    source_path   TEXT NOT NULL,
    embedding_id  INTEGER UNIQUE NOT NULL,
    created_at    TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
CREATE INDEX idx_chunks_primary_law  ON chunks(primary_law);
CREATE INDEX idx_chunks_subtopic     ON chunks(subtopic);
CREATE INDEX idx_chunks_kind         ON chunks(kind);
CREATE INDEX idx_chunks_category     ON chunks(category);
CREATE INDEX idx_chunks_embedding_id ON chunks(embedding_id);

CREATE TABLE chunk_laws (
    chunk_id      INTEGER NOT NULL,
    law_name      TEXT NOT NULL,
    article_no    INTEGER NOT NULL,
    article_full  TEXT NOT NULL,
    paragraph     TEXT,
    role          TEXT NOT NULL,
    FOREIGN KEY (chunk_id) REFERENCES chunks(id) ON DELETE CASCADE
);
CREATE INDEX idx_cl_law   ON chunk_laws(law_name, article_no);
CREATE INDEX idx_cl_chunk ON chunk_laws(chunk_id);
CREATE INDEX idx_cl_role  ON chunk_laws(role);

CREATE TABLE violations (
    id            INTEGER PRIMARY KEY AUTOINCREMENT,
    year          INTEGER NOT NULL,
    month         INTEGER NOT NULL,
    date          TEXT,
    product       TEXT,
    channel       TEXT,
    violation     TEXT NOT NULL,
    company       TEXT,
    penalty_twd   INTEGER,
    law_cited     TEXT,
    article_no    INTEGER,
    source_file   TEXT,
    embedding_id  INTEGER UNIQUE NOT NULL
);
CREATE INDEX idx_viol_article ON violations(article_no);
CREATE INDEX idx_viol_company ON violations(company);
CREATE INDEX idx_viol_penalty ON violations(penalty_twd);
CREATE INDEX idx_viol_date    ON violations(year, month);

CREATE TABLE failed_files (
    id              INTEGER PRIMARY KEY AUTOINCREMENT,
    source_path     TEXT NOT NULL,
    file_name       TEXT NOT NULL,
    file_ext        TEXT,
    page_count      INTEGER,
    failure_reason  TEXT NOT NULL,
    inferred_law    TEXT,
    inferred_topic  TEXT,
    note            TEXT,
    detected_at     TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
CREATE INDEX idx_failed_reason ON failed_files(failure_reason);
"""

# ── 工具函式 ──────────────────────────────────────
def load_jsonl(name: str) -> list:
    p = Path(name)
    if not p.exists():
        print(f"  ⚠️  {name} 不存在，跳過")
        return []
    out = []
    with p.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                out.append(json.loads(line))
    return out

def encode_texts(model, texts, batch_size, desc):
    if not texts:
        return np.zeros((0, model.get_sentence_embedding_dimension()), dtype=np.float32)
    vecs = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    return vecs.astype(np.float32)

def build_faiss(vectors):
    if vectors.shape[0] == 0:
        return faiss.IndexFlatIP(1024)
    idx = faiss.IndexFlatIP(vectors.shape[1])
    idx.add(vectors)
    return idx

# ── 讀資料 ────────────────────────────────────────
print("讀取 JSONL...")
chunks     = load_jsonl("chunks.jsonl")
violations = load_jsonl("violations.jsonl")
failed     = load_jsonl("failed_files.jsonl")

print(f"  chunks     : {len(chunks)}")
print(f"  violations : {len(violations)}")
print(f"  failed     : {len(failed)}")

# 去重
seen, deduped = set(), []
for c in chunks:
    t = c["text"].strip()
    if t not in seen:
        seen.add(t)
        deduped.append(c)
if len(deduped) < len(chunks):
    print(f"  去重後 chunks: {len(deduped)}（移除 {len(chunks)-len(deduped)} 筆）")
chunks = deduped

# ── 載入模型 ──────────────────────────────────────
print(f"\n載入 {EMBED_MODEL}...")
model = SentenceTransformer(EMBED_MODEL, device=EMBED_DEVICE)

# ── 編碼 ──────────────────────────────────────────
print(f"\n[1/2] 編碼 chunks ({len(chunks)} 筆)...")
chunk_vecs = encode_texts(model, [c["text"] for c in chunks], EMBED_BATCH, "chunks")

print(f"\n[2/2] 編碼 violations ({len(violations)} 筆)...")
viol_vecs  = encode_texts(model, [v["violation"] for v in violations], EMBED_BATCH, "violations")

# ── 寫 SQLite ─────────────────────────────────────
print(f"\n寫入 SQLite: {DB_PATH}")
if DB_PATH.exists():
    DB_PATH.unlink()
conn = sqlite3.connect(str(DB_PATH))
conn.executescript(SCHEMA_SQL)
conn.commit()
cur = conn.cursor()

for i, c in enumerate(tqdm(chunks, desc="寫 chunks")):
    cur.execute(
        """INSERT INTO chunks (
            text, category, primary_law, subtopic, document, kind,
            is_ocr, has_table, char_len, chunk_strategy, source_path, embedding_id
        ) VALUES (?,?,?,?,?,?,?,?,?,?,?,?)""",
        (
            c["text"], c.get("category"), c.get("primary_law"),
            c.get("subtopic"), c.get("document"), c.get("kind"),
            1 if c.get("is_ocr") else 0,
            1 if c.get("has_table") else 0,
            c.get("char_len"), c.get("chunk_strategy"),
            c.get("source_path"), i,
        ),
    )
    chunk_id = cur.lastrowid
    for law in c.get("law_refs", []):
        cur.execute(
            """INSERT INTO chunk_laws (
                chunk_id, law_name, article_no, article_full, paragraph, role
            ) VALUES (?,?,?,?,?,?)""",
            (chunk_id, law["law_name"], law["article_no"],
             law["article_full"], law.get("paragraph"), law["role"]),
        )

for i, v in enumerate(tqdm(violations, desc="寫 violations")):
    cur.execute(
        """INSERT INTO violations (
            year, month, date, product, channel, violation,
            company, penalty_twd, law_cited, article_no, source_file, embedding_id
        ) VALUES (?,?,?,?,?,?,?,?,?,?,?,?)""",
        (
            v["year"], v["month"], v.get("date"), v.get("product"),
            v.get("channel"), v["violation"], v.get("company"),
            v.get("penalty_twd"), v.get("law_cited"), v.get("article_no"),
            v.get("source_file"), i,
        ),
    )

for f in failed:
    cur.execute(
        """INSERT INTO failed_files (
            source_path, file_name, file_ext, page_count,
            failure_reason, inferred_law, inferred_topic, note
        ) VALUES (?,?,?,?,?,?,?,?)""",
        (
            f.get("source_path"), f.get("file_name"), f.get("file_ext"),
            f.get("page_count"), f.get("failure_reason"),
            f.get("inferred_law"), f.get("inferred_topic"), f.get("note"),
        ),
    )

conn.commit()
conn.close()
print(f"SQLite 完成，檔案大小: {DB_PATH.stat().st_size / 1e6:.1f} MB")

# ── 寫 FAISS ──────────────────────────────────────
print(f"\n寫入 FAISS chunks index...")
faiss.write_index(build_faiss(chunk_vecs), str(FAISS_CHUNKS))
print(f"寫入 FAISS cases index...")
faiss.write_index(build_faiss(viol_vecs),  str(FAISS_CASES))

print("\n" + "="*50)
print("✅ Step 2 完成！")
print(f"   chunks.db          : {DB_PATH.stat().st_size/1e6:.1f} MB")
print(f"   faiss_chunks.index : {FAISS_CHUNKS.stat().st_size/1e6:.1f} MB")
print(f"   faiss_cases.index  : {FAISS_CASES.stat().st_size/1e6:.1f} MB")
print("\n接著跑下一格下載檔案")

In [ ]:
# ⑤ 下載結果回本機
# 下載後放到本機專案的 data/index/ 資料夾
from google.colab import files

print("下載中...（瀏覽器會跳出三個下載）")
files.download("chunks.db")
files.download("faiss_chunks.index")
files.download("faiss_cases.index")
print("下載完成！把三個檔案放到本機的 data/index/ 資料夾，然後 make run 啟動 API。")